In [1]:
import torch
import numpy as np
# import scipy.special as sp

# import pickle as pkl
# import zlib
# import base64

/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.path.append('/Users/aleksei/projects/code-of-kutulu-client')

In [3]:
from src.envs.agents.reinforce_agent import REINFORCEAgent
from src.envs.agents.dqn_agent_ext import DQNAgentExt
from src.envs.agents.dqn_agent import DQNAgent

In [4]:
# agent = REINFORCEAgent(**{
#     'state_type': 'closest_ext',
#     'gamma': 0.5,
#     'action_space_n': 8,
#     'train': False,
# })

In [32]:
agent = DQNAgentExt(**{
    'state_type': 'closest_ext',
    'gamma': 0.5,
    'action_space_n': 4,
    'train': False,
})

In [33]:
checkpoints_dir = '../output/2025-05-24/22:32:15.793442/agent2'
agent.load_agent(checkpoints_dir)

In [34]:
# agent = DQNAgent(**{
#     'state_type': 'closest',
#     'gamma': 0.5,
#     'action_space_n': 5,
#     'train': False,
# })
# checkpoints_dir = '../output/2025-05-24/22:32:15.793442/agent2'
# agent.load_agent(checkpoints_dir)

In [35]:

from src.envs.distance import find_path
from src.envs.kutulu_observer import KutuluClosestObserver, KutuluClosestExtObserver
from src.envs.kutulu_world import KutuluWorldEnv
from src.game.template import CELL_WALL, DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS
from src.game.template import MOVE_REL_POS, REL_POSITIONS

In [36]:
env = KutuluWorldEnv('', '', 1, actions=EXTENDED_KUTULU_ACTIONS)
env.map = [
    #0123456
    '#######', # 0
    '#.....#', # 1
    '#.#.#.#', # 2
    '#.....#', # 3
    '#.#.#.#', # 4
    '#.....#', # 5
    '#######', # 6
]
env.width = len(env.map[0])
env.height = len(env.map)

In [37]:
all_answers = set(range(4))
params = [
    (set([answer]), [rel_pos], [])
    for answer, rel_pos in enumerate(REL_POSITIONS[:-1])
] + [
    (set([answer]), [(tuple(x * 2 for x in rel_pos))], [])
    for answer, rel_pos in enumerate(REL_POSITIONS[:-1])
] + [
    (set([answer]), [(tuple(x * 3 for x in rel_pos))], [])
    for answer, rel_pos in enumerate(REL_POSITIONS[:-1])
] + [
    (all_answers - set([answer]), [], [rel_pos])
    for answer, rel_pos in enumerate(REL_POSITIONS[:-1])
] + [
    (all_answers - set([answer]), [], [tuple(x * 2 for x in rel_pos)])
    for answer, rel_pos in enumerate(REL_POSITIONS[:-1])
]

In [38]:
def set_env(agent, env, explorers, wanderers):
    player_id = 0
    player_pos = (3, 3)
    obs = [
        None,
        f'EXPLORER 0 {player_pos[0]} {player_pos[1]} 100 0 0'
    ] + [
        f'EXPLORER {i + 1} {player_pos[0] + x} {player_pos[1] + y} 10 0 0'
        for i, (x, y) in enumerate(explorers) 
    ] + [
        f'WANDERER {i + 10} {player_pos[0] + x} {player_pos[1] + y} 10 1 0'
        for i, (x, y) in enumerate(wanderers) 
    ]
    env._set_entities(obs)
    env._set_players(obs, set_ids=True)
    agent.set_env(env)

In [39]:
for answer, explorers, wanderers in params:
    set_env(agent, env, explorers, wanderers)
    state, action = agent.generate_state_and_step(0)
    print(action, answer, explorers, wanderers)

1 {0} [(0, -1)] []
1 {1} [(1, 0)] []
1 {2} [(0, 1)] []
1 {3} [(-1, 0)] []
1 {0} [(0, -2)] []
1 {1} [(2, 0)] []
1 {2} [(0, 2)] []
1 {3} [(-2, 0)] []
1 {0} [(0, -3)] []
1 {1} [(3, 0)] []
1 {2} [(0, 3)] []
1 {3} [(-3, 0)] []
3 {1, 2, 3} [] [(0, -1)]
3 {0, 2, 3} [] [(1, 0)]
3 {0, 1, 3} [] [(0, 1)]
1 {0, 1, 2} [] [(-1, 0)]
3 {1, 2, 3} [] [(0, -2)]
3 {0, 2, 3} [] [(2, 0)]
3 {0, 1, 3} [] [(0, 2)]
1 {0, 1, 2} [] [(-2, 0)]


In [40]:
set_env(agent, env, [(0, -2)], [])

In [41]:
self = agent

In [42]:
player_id = 0

In [43]:
valid_actions = self.get_valid_actions(player_id)
player_mask = ~np.array(valid_actions)
player_mask = player_mask[:self.action_space_n]


state = self.get_state(player_id)
data = self.episode_buffer.encode_states([state])
model_output = self.model(data)[0].detach().cpu().numpy()
actions_masked = np.ma.array(model_output, mask=player_mask)

In [44]:
model_output.std()

0.001630336

In [45]:
model_output

array([1.4673356, 1.4718418, 1.4687765, 1.4695463], dtype=float32)

In [46]:
actions_masked.std()

0.0016303361211264533

In [47]:
(actions_masked / actions_masked.sum()).std()

0.00027738225114762083